<a href="https://colab.research.google.com/github/oszavla/Dirac-Schwarz/blob/main/Unitary_Similarity_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# 1. Definisikan matriks Pauli 2x2 standar yang dikalikan dengan imajiner negatif (-i) untuk tanda signature -,+,+,+
sigma1 = np.array([[0, -1j], [-1j, 0]], dtype=complex)
sigma2 = np.array([[0, -1], [1, 0]], dtype=complex)
sigma3 = np.array([[-1j, 0], [0, 1j]], dtype=complex)
I_2 = np.eye(2, dtype=complex)
zeros = np.zeros((2, 2), dtype=complex)

# 2. Definisikan matriks Gamma Dirac 4x4 (representasi Dirac)
gamma0 = np.block([[-1j * I_2, zeros], [zeros, 1j * I_2]])
gamma1 = np.block([[zeros, sigma1], [-sigma1, zeros]])
gamma2 = np.block([[zeros, sigma2], [-sigma2, zeros]])
gamma3 = np.block([[zeros, sigma3], [-sigma3, zeros]])

I_4 = np.eye(4, dtype=complex)

# Simpan matriks ke dalam list untuk iterasi yang mudah
gammas = [gamma0, gamma1, gamma2, gamma3]

# 3. Definisikan matriks a dan a_inv menggunakan operator @ untuk perkalian matriks
a = 0.5 * (gamma1 @ gamma2 - gamma1 @ gamma3 + gamma2 @ gamma3 + I_4)
a_inv = 0.5 * (I_4 - gamma1 @ gamma2 + gamma1 @ gamma3 - gamma2 @ gamma3)

# Verifikasi apakah a @ a_inv benar-benar merupakan matriks identitas
identity_check = np.allclose(a @ a_inv, I_4)
print(f"Apakah a @ a_inv sama dengan matriks identitas? {identity_check}\n")

# 4. Hitung matriks yang ditransformasi: gamma'_mu = a @ gamma_mu @ a_inv
gamma_primes = []
for mu in range(4):
    gamma_prime = a_inv @ gammas[mu] @ a
    gamma_primes.append(gamma_prime)

# 5. Fungsi pembantu untuk mengidentifikasi matriks hasil
def identify_matrix(mat, gammas):
    """Membandingkan matriks yang ditransformasi dengan matriks gamma asli."""
    # Kita memeriksa apakah hasilnya cocok dengan salah satu dari +/- gamma^nu
    for nu, g in enumerate(gammas):
        if np.allclose(mat, g):
            return f"+ gamma^{nu}"
        elif np.allclose(mat, -g):
            return f"- gamma^{nu}"
    return "Kombinasi linear kompleks (tidak memetakan ke satu matriks gamma saja)"

# Cetak hasil
print("Hasil dari a_inv @ gamma^mu @ a:")
print("-" * 40)
for mu in range(4):
    identity = identify_matrix(gamma_primes[mu], gammas)
    print(f"gamma^'{mu} = {identity}")

Is a @ a_inv equal to the identity matrix? True

Results of a_inv @ gamma^mu @ a:
----------------------------------------
gamma^'0 = + gamma^0
gamma^'1 = + gamma^2
gamma^'2 = + gamma^3
gamma^'3 = + gamma^1


In [ ]:
from sys import path_importer_cache
import sympy as sp
from sympy.physics.matrices import msigma

# 1. Define the rotation angle parameter
theta, phi = sp.symbols('theta phi', real=True, positive=True)

# 2. Grab the 2x2 Pauli matrices and identity
sigma_1 = msigma(1)
sigma_2 = msigma(2)
sigma_3 = msigma(3)
I2 = sp.eye(2)
Z2 = sp.zeros(2)
sigmas = [sigma_1, sigma_2, sigma_3]

# for i in range(3):
#   display(sigmas[i])

# 3. Construct the 4x4 Dirac Gamma Matrices
# (Using the Standard Dirac-Pauli Representatio multiplied by negative iaginary (-i) for signature -,+,+,+
# Note: BlockMatrix automatically stitches the 2x2 matrices into 4x4s
gamma_0 = sp.Matrix(sp.BlockMatrix([[(- sp.I) * I2, Z2], [Z2, - sp.I * (-I2)]]))
gamma_1 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_1], [(- sp.I) * (-sigma_1), Z2]]))
gamma_2 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_2], [(- sp.I) * (-sigma_2), Z2]]))
gamma_3 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_3], [(- sp.I) * (-sigma_3), Z2]]))
I4      = sp.eye(4)

gamma_a = [gamma_0, gamma_1, gamma_2, gamma_3]

# 4. Define your operator A and target matrix B
# We replicate the spatial local Lorentz rotation: A = -(\theta/2) * \gamma^3  * \gamma^1
T = -(theta / 2) * (gamma_3 * gamma_1)
P = -(phi / 2) * (gamma_1 * gamma_2)

# 5. Compute the exact matrix exponential transformation
exp_T = sp.exp(T)
inv_exp_T = sp.exp(-T)
exp_P = sp.exp(P)
inv_exp_P = sp.exp(-P)

print(f"--------- ( Lambda gamma_[a] inv_Lambda )  --------")

for a in range(4):
    gamma_a[a] = exp_P * (exp_T * gamma_a[a] * inv_exp_T) * inv_exp_P
    gamma_a[a] = sp.simplify(gamma_a[a])
    print(f"Lambda gamma_[{a}] inv_Lambda = {gamma_a[a]}")
    print(f"Matrix Form Lambda gamma_[{a}] inv_Lambda =")
    display(gamma_a[a])

print(f"---------inv_a inv_Lambda ( Lambda gamma_[a] inv_Lambda ) Lambda a  --------")

# 3. Define the a and a_inv matrices using the * operator for matrix multiplication
a     = 0.5 * (  gamma_1 * gamma_2 - gamma_1 * gamma_3 + gamma_2 * gamma_3 + I4)
a_inv = 0.5 * (- gamma_1 * gamma_2 + gamma_1 * gamma_3 - gamma_2 * gamma_3 + I4)

for a in range(4):
    gamma_a[a] = a_inv * ( inv_exp_T * (inv_exp_P * gamma_a[a] * exp_P) * exp_T ) * a
    gamma_a[a] = sp.simplify(gamma_a[a])
    print(f"inv_a inv_Lambda ( Lambda gamma_[{a}] inv_Lambda ) Lambda a = {gamma_a[a]}")
    print(f"Matrix inv_a inv_Lambda ( Lambda gamma_[{a}] inv_Lambda ) Lambda a =")
    display(gamma_a[a])

--------- ( Lambda gamma_[a] inv_Lambda )  --------
Lambda gamma_[0] inv_Lambda = Matrix([[-I, 0, 0, 0], [0, -I, 0, 0], [0, 0, I, 0], [0, 0, 0, I]])
Matrix Form Lambda gamma_[0] inv_Lambda =


Matrix([
[-I,  0, 0, 0],
[ 0, -I, 0, 0],
[ 0,  0, I, 0],
[ 0,  0, 0, I]])

Lambda gamma_[1] inv_Lambda = Matrix([[0, 0, I*sin(theta), -I*exp(-I*phi)*cos(theta)], [0, 0, -I*exp(I*phi)*cos(theta), -I*sin(theta)], [-I*sin(theta), I*exp(-I*phi)*cos(theta), 0, 0], [I*exp(I*phi)*cos(theta), I*sin(theta), 0, 0]])
Matrix Form Lambda gamma_[1] inv_Lambda =


Matrix([
[                      0,                        0,             I*sin(theta), -I*exp(-I*phi)*cos(theta)],
[                      0,                        0, -I*exp(I*phi)*cos(theta),             -I*sin(theta)],
[          -I*sin(theta), I*exp(-I*phi)*cos(theta),                        0,                         0],
[I*exp(I*phi)*cos(theta),             I*sin(theta),                        0,                         0]])

Lambda gamma_[2] inv_Lambda = Matrix([[0, 0, 0, -exp(-I*phi)], [0, 0, exp(I*phi), 0], [0, exp(-I*phi), 0, 0], [-exp(I*phi), 0, 0, 0]])
Matrix Form Lambda gamma_[2] inv_Lambda =


Matrix([
[          0,           0,          0, -exp(-I*phi)],
[          0,           0, exp(I*phi),            0],
[          0, exp(-I*phi),          0,            0],
[-exp(I*phi),           0,          0,            0]])

Lambda gamma_[3] inv_Lambda = Matrix([[0, 0, -I*cos(theta), -I*exp(-I*phi)*sin(theta)], [0, 0, -I*exp(I*phi)*sin(theta), I*cos(theta)], [I*cos(theta), I*exp(-I*phi)*sin(theta), 0, 0], [I*exp(I*phi)*sin(theta), -I*cos(theta), 0, 0]])
Matrix Form Lambda gamma_[3] inv_Lambda =


Matrix([
[                      0,                        0,            -I*cos(theta), -I*exp(-I*phi)*sin(theta)],
[                      0,                        0, -I*exp(I*phi)*sin(theta),              I*cos(theta)],
[           I*cos(theta), I*exp(-I*phi)*sin(theta),                        0,                         0],
[I*exp(I*phi)*sin(theta),            -I*cos(theta),                        0,                         0]])

---------inv_a inv_Lambda ( Lambda gamma_[a] inv_Lambda ) Lambda a  --------
inv_a inv_Lambda ( Lambda gamma_[0] inv_Lambda ) Lambda a = Matrix([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
Matrix inv_a inv_Lambda ( Lambda gamma_[0] inv_Lambda ) Lambda a =
inv_a inv_Lambda ( Lambda gamma_[1] inv_Lambda ) Lambda a = Matrix([[0, 0, -0.5 + 0.5*I, 0.5*I*(-1 + I)], [0, 0, 0.5 - 0.5*I, 0.5*I*(-1 + I)], [0.5 - 0.5*I, 0.5 + 0.5*I, 0, 0], [-0.5 + 0.5*I, 0.5 + 0.5*I, 0, 0]])
Matrix inv_a inv_Lambda ( Lambda gamma_[1] inv_Lambda ) Lambda a =
inv_a inv_Lambda ( Lambda gamma_[2] inv_Lambda ) Lambda a = Matrix([[0, 0, -1.0 - 1.0*I, -1.0 + 1.0*I], [0, 0, 1.0 + 1.0*I, -1.0 + 1.0*I], [1.0 + 1.0*I, 1.0 - 1.0*I, 0, 0], [-1.0 - 1.0*I, 1.0 - 1.0*I, 0, 0]])
Matrix inv_a inv_Lambda ( Lambda gamma_[2] inv_Lambda ) Lambda a =
inv_a inv_Lambda ( Lambda gamma_[3] inv_Lambda ) Lambda a = Matrix([[0, 0, 1.5*I*(-1 + I), 1.5 - 1.5*I], [0, 0, 1.5*I*(-1 + I), -1.5 + 1.5*I], [1.5 + 1.5*I, -1.5 + 1.5*I, 0, 0

In [ ]:
from sys import path_importer_cache
import sympy as sp
from sympy.physics.matrices import msigma

# 1. Define the rotation angle parameter
theta, phi = sp.symbols('theta phi', real=True, positive=True)

# 2. Grab the 2x2 Pauli matrices and identity
sigma_1 = msigma(1)
sigma_2 = msigma(2)
sigma_3 = msigma(3)
I2 = sp.eye(2)
Z2 = sp.zeros(2)
sigmas = [sigma_1, sigma_2, sigma_3]

# for i in range(3):
#   display(sigmas[i])

# 3. Construct the 4x4 Dirac Gamma Matrices
# (Using the Standard Dirac-Pauli Representatio multiplied by negative iaginary (-i) for signature -,+,+,+
# Note: BlockMatrix automatically stitches the 2x2 matrices into 4x4s
gamma_0 = sp.Matrix(sp.BlockMatrix([[(- sp.I) * I2, Z2], [Z2, - sp.I * (-I2)]]))
gamma_1 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_1], [(- sp.I) * (-sigma_1), Z2]]))
gamma_2 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_2], [(- sp.I) * (-sigma_2), Z2]]))
gamma_3 = sp.Matrix(sp.BlockMatrix([[Z2, (- sp.I) * sigma_3], [(- sp.I) * (-sigma_3), Z2]]))
I4      = sp.eye(4)

gamma_a = [gamma_0, gamma_1, gamma_2, gamma_3]

# 4. Define your operator A and target matrix B
# We replicate the spatial local Lorentz rotation: A = -(\theta/2) * \gamma^3  * \gamma^1
T = -(theta / 2) * (gamma_3 * gamma_1)
P = -(phi / 2) * (gamma_1 * gamma_2)

# 5. Compute the exact matrix exponential transformation
exp_T = sp.exp(T)
inv_exp_T = sp.exp(-T)
exp_P = sp.exp(P)
inv_exp_P = sp.exp(-P)


# 3. Define the a and a_inv matrices using the * operator for matrix multiplication
a     = 0.5 * (  gamma_1 * gamma_2 - gamma_1 * gamma_3 + gamma_2 * gamma_3 + I4)
a_inv = 0.5 * (- gamma_1 * gamma_2 + gamma_1 * gamma_3 - gamma_2 * gamma_3 + I4)

print(f"---------(2)--------")
gammaNew2 = a_inv * ( inv_exp_T * (gamma_3 * gamma_1) * exp_T ) * a
gammaNew2 = sp.simplify(gamma_2 * gammaNew2)
display(gammaNew2)

print(f"---------(3)--------")
gammaNew3 = a_inv * ( inv_exp_T * (inv_exp_P * (gamma_1 * gamma_2) * exp_P) * exp_T ) * a
gammaNew3 = sp.simplify(gamma_3 * gammaNew3)
display(gammaNew3)

---------(2)--------


Matrix([
[     0,      0,     0, 1.0*I],
[     0,      0, 1.0*I,     0],
[     0, -1.0*I,     0,     0],
[-1.0*I,      0,     0,     0]])

---------(3)--------


Matrix([
[                0,                 0,                  0, 1.0*exp(I*theta)],
[                0,                 0, -1.0*exp(-I*theta),                0],
[                0, -1.0*exp(I*theta),                  0,                0],
[1.0*exp(-I*theta),                 0,                  0,                0]])